# JSpaceDoNOT: model preparation, lens fitting, and three experiments

Run the cells in order: **model download → pinned J++ source → WikiText-103 raw download and prompt pool → full J++ fitting if missing → GSM8K/MATH-500/RACE-H preparation → experiments → interpretation**.

Existing models, data, and a matching completed lens are reused. Without a completed lens, preparation runs the original full fit. Lens fitting uses seed 0; stopping experiments use seed 2026. The J++ recipe retains 1,000 router prompts, 64 Jacobian prompts, and all 35 source blocks. Full fitting can take substantial time.

Install the project dependencies first and use a GPU environment that supports Qwen3 in BF16. This notebook does not install or downgrade PyTorch, CUDA, or Transformers. Set `FIT_PYTHON` if you have a separate existing J++ environment.

In [ ]:
from pathlib import Path
import sys, importlib.util
from dataclasses import replace

# The project directory must contain jspace_donot/, scripts/, and run.py.
PROJECT_ROOT = None  # Or Path("/path/to/JSpaceDoNoT_GitHub")
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd()/"JSpaceDoNoT_GitHub", Path.cwd()/"JSpaceDoNoT"]
    hits = [p for p in candidates if (p/"scripts/fit_jpp.py").is_file()]
    if not hits:
        spec = importlib.util.find_spec("jspace_donot")
        if spec and spec.origin:
            hits = [Path(spec.origin).resolve().parent.parent]
    if len(hits) != 1:
        raise FileNotFoundError("Set PROJECT_ROOT to the extracted project directory.")
    PROJECT_ROOT = hits[0]
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
assert (PROJECT_ROOT/"scripts").is_dir()
sys.path.insert(0, str(PROJECT_ROOT))
from jspace_donot import prepare_assets, ExperimentConfig, run_experiment

# Root for downloads, fitted assets, and outputs. A separate directory can isolate a new run.
ROOT = PROJECT_ROOT
MODEL_DIR = None       # None -> ROOT/base_models/Qwen3-4B; or an existing model directory
LENS_PATH = None       # None -> fit if missing; or a completed lens file/directory
FIT_PYTHON = None      # Or Path("/your/jpp-env/bin/python"); None uses the current kernel Python
LAYERS_PER_GROUP = 4
JACOBIAN_ROWS_PER_PASS = 8  # Use 4 or 2 to reduce memory; prompt counts, layers, and seed stay fixed
SCHEDULE = "wait"
EVERY_N_TOKENS = 256
CACHE_MODE = "resume"  # resume | read_only | off


## 1. Prepare the model, corpus, lens, and task datasets

This cell downloads missing assets and fits the lens when needed. The corpus is **`Salesforce/wikitext`, `wikitext-103-raw-v1`, TRAIN**. The original selection keeps the first 1,000 records with `len(text.strip()) >= 600`, preserving their original text and order. It does not use the older plain J-lens shuffled prompt pool.

Fitting stages: activation routers → LRP expert Jacobians → signed expert weights → combined lens and manifest. Preparation returns the actual completed `.pt` path automatically.

In [ ]:
assets = prepare_assets(
    root=ROOT,
    datasets=("gsm8k", "math500", "raceh"),
    model_dir=MODEL_DIR,
    lens_path=LENS_PATH,
    fit_if_missing=True,
    download_wikitext=True,
    layers_per_group=LAYERS_PER_GROUP,
    jacobian_rows_per_pass=JACOBIAN_ROWS_PER_PASS,
    fit_python=FIT_PYTHON,
)
ROOT = assets.root
LENS_PATH = assets.lens_path
print("Model:", assets.model_dir)
print("WikiText pool:", assets.wikitext_prompts)
print("Completed J++ lens:", LENS_PATH)


## 2. Run the three experiments with the original settings

Each experiment defaults to 256 TRAIN questions and the complete TEST split. Four-fold calibration, candidate construction, and classifier settings retain the original implementation. Use `SCHEDULE`, `EVERY_N_TOKENS`, and `CACHE_MODE` to configure monitoring and caching. Each task runs in a separate worker process, releasing its GPU model when it finishes.

In [ ]:
common = ExperimentConfig(
    dataset="gsm8k", root=ROOT, lens_path=LENS_PATH,
    schedule=SCHEDULE, every_n_tokens=EVERY_N_TOKENS,
    cache_mode=CACHE_MODE,
)
configs = {task: replace(common, dataset=task) for task in ("gsm8k", "math500", "raceh")}
results = {}


## GSM8K

In [ ]:
results["gsm8k"] = run_experiment(configs["gsm8k"])
print(results["gsm8k"].output)
display(results["gsm8k"].summary)


## MATH-500

In [ ]:
results["math500"] = run_experiment(configs["math500"])
print(results["math500"].output)
display(results["math500"].summary)


## RACE-H

In [ ]:
results["raceh"] = run_experiment(configs["raceh"])
print(results["raceh"].output)
display(results["raceh"].summary)


## 3. Interpret the fitted stopping heads

Interpretation uses the actual saved LR/MLP heads. It does not refit Qwen or the lens and can run on CPU after restarting the kernel.

In [ ]:
from jspace_donot.interpretation import interpret_run, StopInterpreter
RUN_DIR = results["gsm8k"].output  # After a restart, replace with a completed output directory
output = interpret_run(RUN_DIR, linear_C=.1, mlp_C=1., cutoff=.85,
                       max_ig_examples=300, ig_steps=48)
print(output)


## 4. Optional case studies

Inspect the actual candidate tokens to establish what each feature column represents before defining group interventions. A STOP-to-CONTINUE case search may return no examples.

In [ ]:
RUN_CASE_STUDIES = False
if RUN_CASE_STUDIES:
    explainer = StopInterpreter.load(RUN_DIR)
    cases = explainer.find_stop_flips(cutoff=.85)
    display(cases.head())
    if not cases.empty:
        case = cases.iloc[0]
        raw, context = explainer.checkpoint(case.question_id, int(case.step))
        print(context.get("token_strings"), context.get("cot_prefix"))
        table, _ = explainer.group_ablation(case.question_id, int(case.step),
            {"L3 ranks 1+2":[8,9], "L4 ranks 1+2":[12,13], "Both":[8,9,12,13]})
        display(table)
